# MRT → catalog_pages

Полный `catalog.json` в контекст LLM не кладём.

1. `extract_mrt` читает JSON `.mrt`
2. пишет `catalog.json` + `catalog_pages/`
3. в промпт — только `index.json`, нужные pages — через `N.json`

In [ ]:
from pathlib import Path
import json

from extract_mrt import catalog_from_report, load_mrt, write_page_packs, build_page_index

MRT = Path("SalesOfProducts.mrt")
CATALOG = Path("catalog.json")
PAGES_DIR = Path("catalog_pages")

MRT, MRT.exists(), MRT.stat().st_size

## 1. Extract

In [ ]:
catalog = catalog_from_report(load_mrt(MRT))
CATALOG.write_text(json.dumps(catalog, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
write_page_packs(catalog, PAGES_DIR)

{
    "report": catalog.get("report"),
    "databases": len(catalog.get("databases") or []),
    "sources": len(catalog.get("sources") or []),
    "relations": len(catalog.get("relations") or []),
    "variables": len(catalog.get("variables") or []),
    "pages": len(catalog.get("pages") or []),
    "catalog_bytes": CATALOG.stat().st_size,
}

## 2. Что лежит в `catalog_pages/`

| файл | зачем |
|---|---|
| `index.json` | карточки всех pages → всегда в промпт |
| `dictionary.json` | sources/variables/relations без тел pages |
| `0.json`, `1.json`, … | pack одной page + связанные sources |

In [ ]:
files = sorted(PAGES_DIR.iterdir(), key=lambda p: p.name)
for path in files:
    print(f"{path.name:20} {path.stat().st_size:8} bytes")

index = json.loads((PAGES_DIR / "index.json").read_text(encoding="utf-8"))
index

## 3. Pack одной страницы

В контекст после выбора page — только этот pack, не весь catalog.

In [ ]:
pack = json.loads((PAGES_DIR / "0.json").read_text(encoding="utf-8"))
{
    "index": pack.get("index"),
    "index_entry": pack.get("index_entry"),
    "sources": [s.get("name") for s in pack.get("sources") or []],
    "variables": [v.get("name") for v in pack.get("variables") or []],
    "relations": len(pack.get("relations") or []),
    "elements": [
        {
            "name": e.get("name"),
            "type": e.get("type"),
            "role": e.get("role"),
            "title": e.get("title"),
            "expressions": e.get("expressions"),
        }
        for e in (pack.get("page") or {}).get("elements") or []
    ],
    "pack_bytes": (PAGES_DIR / "0.json").stat().st_size,
}

## 4. Сравнение размеров (что реально влезает в контекст)

In [ ]:
sizes = {
    "mrt": MRT.stat().st_size,
    "catalog.json": CATALOG.stat().st_size,
    "index.json": (PAGES_DIR / "index.json").stat().st_size,
    "dictionary.json": (PAGES_DIR / "dictionary.json").stat().st_size,
    "0.json": (PAGES_DIR / "0.json").stat().st_size,
}
sizes["prompt_min (index only)"] = sizes["index.json"]
sizes["prompt_page (index + 0)"] = sizes["index.json"] + sizes["0.json"]
sizes

## 5. Как кормить LLM

```text
system: index.json (+ variable defaults)
user:   вопрос
tool:   load_page(0) → 0.json
tool:   execute_sql(...)
answer: по данным + логике page
```

Top-K отсечения нет: можно пройтись по всему индексу и подтянуть несколько pages.

In [ ]:
def load_page(ref: int | str) -> dict:
    """Как tool load_page: по index или name/alias."""
    pages = index.get("pages") or []
    if isinstance(ref, int) or (isinstance(ref, str) and ref.isdigit()):
        i = int(ref)
    else:
        needle = str(ref).strip().lower()
        i = next(
            entry["index"]
            for entry in pages
            if str(entry.get("name") or "").lower() == needle
            or str(entry.get("alias") or "").lower() == needle
        )
    return json.loads((PAGES_DIR / f"{i}.json").read_text(encoding="utf-8"))

loaded = load_page("Products")  # alias Dashboard1
loaded["index_entry"], [s["name"] for s in loaded.get("sources") or []]

## 6. Локальная LLM

Контекст = `index.json` + выбранный page pack. Endpoint из `.env`: `LLM_BASE_URL` / `LLM_MODEL`.

In [ ]:
import os
import urllib.error
import urllib.request

def load_dotenv(path: Path = Path(".env")) -> dict[str, str]:
    env: dict[str, str] = {}
    if not path.is_file():
        return env
    for line in path.read_text(encoding="utf-8").splitlines():
        raw = line.strip()
        if not raw or raw.startswith("#") or "=" not in raw:
            continue
        key, value = raw.split("=", 1)
        env[key.strip()] = value.strip().strip('"').strip("'")
    return env

env = load_dotenv()
for key, value in env.items():
    os.environ.setdefault(key, value)

LLM_BASE_URL = os.environ.get("LLM_BASE_URL", "http://127.0.0.1:8080/v1").rstrip("/")
LLM_MODEL = os.environ.get("LLM_MODEL", "").strip()
LLM_API_KEY = os.environ.get("LLM_API_KEY", "").strip()

def llm_request(method: str, path: str, body: dict | None = None, timeout: float = 180.0) -> dict:
    url = f"{LLM_BASE_URL}{path}"
    data = None if body is None else json.dumps(body).encode("utf-8")
    headers = {"Content-Type": "application/json"}
    if LLM_API_KEY:
        headers["Authorization"] = f"Bearer {LLM_API_KEY}"
    req = urllib.request.Request(url, data=data, headers=headers, method=method)
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))

def list_models() -> list[str]:
    payload = llm_request("GET", "/models")
    return [item.get("id") for item in payload.get("data") or [] if item.get("id")]

def chat(messages: list[dict], model: str | None = None, temperature: float = 0.1) -> dict:
    chosen = (model or LLM_MODEL).strip()
    if not chosen:
        ids = list_models()
        if not ids:
            raise RuntimeError("LLM_MODEL пуст и /models ничего не вернул")
        chosen = ids[0]
    return llm_request(
        "POST",
        "/chat/completions",
        {
            "model": chosen,
            "messages": messages,
            "temperature": temperature,
        },
    )

print("LLM_BASE_URL =", LLM_BASE_URL)
print("LLM_MODEL    =", LLM_MODEL or "(auto from /models)")
try:
    models = list_models()
    print("models:", models)
    if not LLM_MODEL and models:
        LLM_MODEL = models[0]
        print("using:", LLM_MODEL)
except Exception as exc:
    print("LLM недоступна:", type(exc).__name__, exc)
    print("Подними локальный OpenAI-compatible сервер на", LLM_BASE_URL)

Собираем промпт: system = index + page pack, user = вопрос. Пока без SQL — смотрим, что модель понимает по структуре дашборда.

In [ ]:
QUESTION = "Какие показатели есть на дашборде и из каких полей/SQL они считаются?"

page_pack = load_page(0)

system_prompt = """Ты аналитик дашборда Stimulsoft.
Ниже индекс pages и полный pack выбранной страницы (элементы, expressions, SQL sources).
Отвечай кратко по-русски только по этому контексту. SQL пока не выполняй — опиши, что есть.
"""

context = {
    "index": index,
    "page_pack": page_pack,
}

messages = [
    {"role": "system", "content": system_prompt + "\n\n" + json.dumps(context, ensure_ascii=False, indent=2)},
    {"role": "user", "content": QUESTION},
]

print("question:", QUESTION)
print("context bytes:", len(messages[0]["content"].encode("utf-8")))
print("sources in pack:", [s.get("name") for s in page_pack.get("sources") or []])

In [ ]:
try:
    response = chat(messages, model=LLM_MODEL or None)
    choice = (response.get("choices") or [{}])[0]
    message = choice.get("message") or {}
    answer = message.get("content") or ""
    print("model:", response.get("model") or LLM_MODEL)
    print("finish:", choice.get("finish_reason"))
    print("usage:", response.get("usage"))
    print("\n=== ANSWER ===\n")
    print(answer)
except urllib.error.URLError as exc:
    print("Не достучался до LLM:", exc)
    print("Запусти сервер на", LLM_BASE_URL, "и повтори ячейку.")
except Exception as exc:
    print(f"{type(exc).__name__}: {exc}")
    if isinstance(exc, urllib.error.HTTPError):
        print(exc.read().decode("utf-8", errors="replace")[:1000])